# dispatch-back-fn-from-recipe — faded example 1: Dispatch loop: iterate parents and look up back functions

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dispatch-back-fn-from-recipe`. Running the beacon reports progress on the `Backprop: dispatch back fn from recipe` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: dispatch back fn from recipe` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dispatch-back-fn-from-recipe`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dispatch-back-fn-from-recipe"
DD_SUBTOPIC = "Backprop: dispatch back fn from recipe"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The dispatch step is the bridge between the recipe (which records the forward call) and the back-function registry (which stores the gradient rules). For each parent in `recipe.parents`, we form a `(func, argnum)` key and retrieve the corresponding back function from the registry. The result is a list of triples ready to be called by the backward-pass driver.

## Faded exercise 1

Complete `dispatch_back_fns`. The `for` loop header is provided; fill in the body — the registry lookup and the append.

**Fill in:** The two-line loop body: look up `back_funcs[(node.recipe.func, argnum)]` and append `(argnum, parent, back_fn)` to `results`.

In [ ]:
def dispatch_back_fns(node, back_funcs):
    results = []
    for argnum, parent in node.recipe.parents.items():
        raise NotImplementedError()  # TODO: The two-line loop body: look up `back_funcs[(node.recipe.func, argnum)]` and append `(argnum, parent, back_fn)` to `results`.
    return results


def _test():
    from dataclasses import dataclass
    from typing import Callable

    @dataclass
    class Recipe:
        func: Callable
        parents: dict

    class FakeTensor:
        def __init__(self, name, recipe=None):
            self.name = name
            self.recipe = recipe

    def mul(a, b): return a * b
    def mul_back_0(g, out, a, b): return g * b
    def mul_back_1(g, out, a, b): return g * a

    a, b = FakeTensor('a'), FakeTensor('b')
    z = FakeTensor('z', Recipe(func=mul, parents={0: a, 1: b}))

    bf = {(mul, 0): mul_back_0, (mul, 1): mul_back_1}
    triples = dispatch_back_fns(z, bf)

    assert len(triples) == 2
    argnum_map = {t[0]: t for t in triples}
    assert argnum_map[0][1] is a
    assert argnum_map[0][2] is mul_back_0
    assert argnum_map[1][1] is b
    assert argnum_map[1][2] is mul_back_1


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def dispatch_back_fns(node, back_funcs):
    results = []
    for argnum, parent in node.recipe.parents.items():
        back_fn = back_funcs[(node.recipe.func, argnum)]
        results.append((argnum, parent, back_fn))
    return results
```
</details>